# R4 dq_check — record `dq_problems` (INT, 0 = healthy)

Runs after the silver pipeline in the `claims_daily` job. Counts **critical** data-quality failures
in `claims.silver.claims_clean` (nulls on the keys and amounts downstream tasks rely on) and records
the total as the task value `dq_problems`, which the `dq_gate` if/else task evaluates:
`dq_problems == 0` → healthy → `publish_gold` runs; otherwise the gate skips it.


In [0]:
dbutils.widgets.text("catalog", "claims")
catalog = dbutils.widgets.get("catalog")

In [0]:
# Critical DQ failures: rows that would poison the gold publish
nulls_df = spark.sql(f"""
SELECT count(*) AS n
FROM {catalog}.silver.claims_clean
WHERE claim_id IS NULL
   OR hospital_id IS NULL
   OR policy_id IS NULL
   OR billed_amount IS NULL
   OR admission_date IS NULL
""")
critical_nulls = nulls_df.first()["n"]

# Silver sanity: pipeline must have produced rows at all
clean_count = spark.sql(f"SELECT count(*) AS n FROM {catalog}.silver.claims_clean").first()["n"]
sanity = 0 if clean_count > 0 else 1

dq_problems = int(critical_nulls) + int(sanity)

print(f"critical_nulls={critical_nulls}, clean_count={clean_count}")
print(f"dq_problems={dq_problems} (0 = healthy)")

In [0]:
# Record the value for the dq_gate if/else task: {{tasks.dq_check.values.dq_problems}} == 0
# (task values can only be set inside a job run — guarded for interactive testing)
try:
    dbutils.jobs.taskValues.set(key="dq_problems", value=dq_problems)
    print("task value 'dq_problems' recorded for dq_gate")
except Exception as e:
    print(f"Not in a job context, task value skipped: {e}")